Importing all the necessary libraries and packages

In [20]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

Using os to access the exercise.py file and the dataset file, as well as their respective directories, and using pandas to read the dataset file

In [21]:
notebook_dir = os.getcwd() # Navigating to the notebook directory, which contains the Jupyter Notebook
data_dir = os.path.normpath(os.path.join(notebook_dir, "..", "data")) #Getting the normal path of the data directory

# Using the pandas library, as well as the command read_csv, to read the dataset file
df = pd.read_csv(os.path.join(data_dir, 'WA_Fn-UseC_-Telco-Customer-Churn.csv'))

Cleaning the data by converting 'TotalCharges' column values from string to numericals, replacing missing values with the corresponding median value and removing any duplicates

In [22]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce') # Used to convert TotalCharges from a string to a numeric value
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median()) # Used to replace any missing values in 'TotalCharges' with the median value
df = df.drop_duplicates() # Safely drops any duplicate rows if they exist in the data

columns_to_clean = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in columns_to_clean:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

Detecting outliers within the dataset (Exploratory Data Analysis)

In [23]:
outliers = {'customers': (df, ['tenure', 'MonthlyCharges', 'TotalCharges'])}

for key, (data, columns) in outliers.items():
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        num_outliers = len(data[(data[col] < lower_bound) | (data[col] > upper_bound)])
        print(f"Column '{col}' has {num_outliers} outliers.")

Column 'tenure' has 0 outliers.
Column 'MonthlyCharges' has 0 outliers.
Column 'TotalCharges' has 0 outliers.


Performing feature engineering by adding 5 new features

In [24]:
# New Feature #1: Adding a counter functionality for the total number of services a customer uses
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = (df[services] == 'Yes').sum(axis=1)

# New Feature #2: Calculating the Average Monthly Charge Per Service
df['AvgChargePerService'] = df['MonthlyCharges'] / (df['TotalServices'] + 1)

# New Feature #3: Checks if a customer is a senior with no partner or dependents and separates those citizens
df['IsSeniorAlone'] = ((df['SeniorCitizen'] == 1) & (df['Partner'] == 'No') & (df['Dependents'] == 'No')).astype(int)

# New Feature #4: Checks the Contract duration and converts it to a binary value (0 or 1)
df['LongTermContract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

# New Feature #5: Checks streaming medium and media
df['UsesStreamingServices'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)

print(df[['TotalServices', 'AvgChargePerService', 'IsSeniorAlone', 'LongTermContract', 'UsesStreamingServices']].head()) # Printing the first five rows of the five new features
print("The data has been successfully cleaned and feature engineering has been implemented.")

   TotalServices  AvgChargePerService  IsSeniorAlone  LongTermContract  \
0              1              14.9250              0                 0   
1              3              14.2375              0                 1   
2              3              13.4625              0                 0   
3              3              10.5750              0                 1   
4              1              35.3500              0                 0   

   UsesStreamingServices  
0                      0  
1                      0  
2                      0  
3                      0  
4                      0  
The data has been successfully cleaned and feature engineering has been implemented.


Splitting and encoding the data

In [25]:
y = df['Churn'].map({'Yes': 1, 'No': 0}) # Maps 'Yes' and 'No' values in the 'Churn' column to 1 and 0 respectively
X = df.drop(columns=['customerID', 'Churn']) # Drops specific identifiers before numerical vector transformations

# Used to convert categorical variables into numeric columns
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# Splitting the data rows for developing the model and for evaluation
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.20, random_state=42, stratify=y)

Using StandardScaler to prevent data leakage

In [26]:
scaler = StandardScaler() # Initializing the scaler object 
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Training the LogisticRegression model and evaluating model performance

In [27]:
# Initializing the LogisticRegression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled) # Used to start the model's evaluation
print(f"Model Validation Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Validation Accuracy: 79.84%
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1035
           1       0.64      0.54      0.59       374

    accuracy                           0.80      1409
   macro avg       0.74      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409

